<div style="padding: 20px 24px; border-radius: 12px; background: linear-gradient(110deg, #12304a, #187b8f); color: white; margin-bottom: 16px;">
  <div style="font-size: 0.78em; letter-spacing: 0.12em; text-transform: uppercase; opacity: 0.85;">HydroMT-SFINCS | Example 06</div>
  <h1 style="color: white; margin: 8px 0 4px;">SFINCS results: making an animation</h1>
  <div style="opacity: 0.95;">Create an animation of water levels and depths from SFINCS output.</div>
</div>


The example used in this notebook is based on a regular SFINCS model, i.e. no subgrid tables are used. In the absence of the subgrid tables, SFINCS computes the water depth by simply substracting the bed levels from the water levels. The (maximum) water depth **h(max)** is stored in the NetCDF output (*sfincs_map.nc*). 

For models without subgrid tables, maximum water depth is available directly from the SFINCS output; see [Example 05A](5a_plot_results_hmax.ipynb). For subgrid models, first downscale maximum water levels using a high-resolution elevation dataset, as shown in [Example 05B](5b_plot_downscaled_results_hmax.ipynb). Apply the same per-timestep downscaling approach when making animations of subgrid results.

The workflow is organized into these steps:

1. Read the model results.
2. Plot instantaneous water depths on the model grid.
3. Inspect water depth at a gauge location.
4. Prepare and create an animation.


Import dependencies


In [ ]:
import geopandas as gpd
import xarray as xr

from hydromt_sfincs import SfincsModel, plots

We use the [cartopy](https://scitools.org.uk/cartopy/docs/latest/) package to plot maps. This packages provides a simple interface to plot geographic data and add background satellite imagery.

### Step 1: Read model results

The model results in sfincs_map.nc are saved as in a staggered grid format, see [SGRID convention](https://publicwiki.deltares.nl/display/NETCDF/Deltares+proposal+for+Staggered+Grid+data+model+(SGRID)). Here we show how to retrieve the face values and translate the dimensions from node indices (m, n) to (x, y) coordinates in order to plot the results on a map.

In [ ]:
# `sfincs_root` is the (relative) path to sfincs root.
sfincs_root = "sfincs_compound"
mod = SfincsModel(sfincs_root, mode="r")

In [ ]:
# we can simply read the model results (sfincs_map.nc and sfincs_his.nc) using the output.read method
mod.output.read()

### Step 2: Plot instantaneous water depths over time

When we are working with a model without subgrid, the output does contain (instantanuous) water depths. However, for subgrid models, we first have to create downscaled water depths using the modeled water levels and the high-resolution bed levels [see notebook on downscaling water depths](5b_plot_downscaled_results_hmax.ipynb). To speed-up the creation of the animation, we only plot every .... th timestep of the results. In addition, in the absence of modeled water depths (so for subgrid models), we approximate the water depth using the water levels and (coarse) bed levels from the model. Replace that with your downscaled water depths!

In [ ]:
time_step = 4

# h from sfincs_map contains the water depths for each cell face. If not present, we approximate it from the surface and bed elevations.
if "h" not in mod.output.data:
    mod.output.data["h"] = mod.output.data["zs"] - mod.output.data["zb"]

# select only positive water depths
h = mod.output.data["h"].where(mod.output.data["h"] > 0)
h.attrs.update(long_name="water depth", unit="m")

# here we plot the water level every time_step-th hour
plots.plot_map_timesteps(h, time_step=time_step, vmax=4)

### Step 3: Inspect water depth at a gauge

Another way to look at the water depths over time, are the timeseries that were created by the model at pre-defined observation stations. 



In [ ]:
# point_h contains the water depths at the sfincs.obs gauge locations
# see mod.plot_basemaps (or next figure) for the location of the observation points
# plot the water level at the specified gauge

station_id = 1

# Approximate water depth at the observation points (when not available for subgrid model)
if "point_h" not in mod.output.data:
    mod.output.data["point_h"] = mod.output.data["zs"] - mod.output.data["zb"]

h_point = mod.output.data["point_h"].rename({"stations": "station_id"})
h_point["station_id"] = h_point["station_id"].astype(int)

_ = h_point.sel({"station_id": station_id}).plot.line(
    x="time",
)

### Step 4: Create animation

An animation is also simple to make with `matplotlib.animation` method. Here we add the surface water level in blue colors next to the overland flood depth with viridis colormap. To make the water depth plots/animations more informative, we can again mask out permanent water or small water depths.

In [ ]:
# filter the water depth data to only include values above a minimum threshold
hmin = 0.05
da_h = mod.output.data["h"].copy()
da_h = da_h.where(da_h > hmin)
da_h.attrs.update(long_name="flood depth", unit="m")

#### Configure and render the animation

Create an animation of the water depths over time. The configurations of the animation are specified in the cell below, change accordingly

In [ ]:
# Configuration of the animation

step = 1  # `step = 1` means one frame every `<step>` dtout.
interval = 250  # interval between frames in milliseconds
vmin = 0  # minimum value for the color scale
vmax = 3  # maximum value for the color scale
cmap = "viridis"  # colormap for the animation
alpha = 1.0  # transparency for the animation
save = False  # whether to save the animation as a video file
return_html = True  # whether to return an HTML animation for inline display

<div style="padding: 14px 18px; border-left: 4px solid #8f1818; background: #f8a1a1; border-radius: 6px; margin: 12px 1em 12px 0; width: calc(100% - 1em); box-sizing: border-box; color: #111111;"><strong><span style="font-size: 1.15em; margin-right: 4px;">&#9888;&#65039;</span> Note: FFmpeg required</strong><p style="margin: 8px 0 0;">The inline preview is a compressed HTML5 video and requires the FFmpeg executable. Saving an MP4 (<code>save = True</code>) also requires FFmpeg. With Conda, install it using <code>conda install -c conda-forge ffmpeg</code>. If you only need a non-MP4 file, set <code>return_html = False</code> to skip the inline preview.</p></div>

In [ ]:
plots.make_animation(
    data=da_h,
    bmap="sat",
    figsize=(11, 7),
    zoomlevel=12,
    step=step,
    interval=interval,
    vmin=vmin,
    vmax=vmax,
    cmap=cmap,
    alpha=alpha,
    title="SFINCS water depth",
    cbar_kwargs={"shrink": 0.6, "anchor": (0, 0)},
    save_path=mod.root.path / "figs" / "sfincs_h.mp4" if save else None,
    return_html=return_html,
)

<div style="padding: 14px 18px; border-left: 4px solid #187b8f; background: #eaf6f8; border-radius: 6px; margin: 12px 1em 12px 0; width: calc(100% - 1em); box-sizing: border-box; color: #111111;"><strong><span style="font-size: 1.15em; margin-right: 4px;">&#128161;</span> NOTE: Animating subgrid results</strong><br><br>For a subgrid animation, downscale the water level (<code>zs</code>) for each timestep separately using the high-resolution elevation data, then assemble the resulting high-resolution depth maps along the time dimension and animate that series. Do not pass the full time series to <code>downscale_floodmap</code> in one call: it reduces non-spatial dimensions to their maximum, which produces an event-maximum map rather than timestep-by-timestep depths.</div>